# 1. IMPORT LIBRARIES

In [52]:
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [2]:
import os

print(os.listdir('/kaggle/input'))

['competitions', 'datasets']


In [3]:
dataset_path = "/kaggle/input/competitions"
print(os.listdir(dataset_path))

['playground-series-s6e6']


In [5]:
import os
dataset_path = "/kaggle/input/datasets/fedesoriano"
print(os.listdir(dataset_path))

['stellar-classification-dataset-sdss17']



# 2. LOAD DATA


In [6]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e6/test.csv')
orig  = pd.read_csv('/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv')


print("Train shape:",train.shape)
print("Test shape:",test.shape)


Train shape: (577347, 12)
Test shape: (247435, 11)


# 3.Label Encoding

In [43]:
le = LabelEncoder()
y = le.fit_transform(y)

# 4. Feature Engineering

In [44]:
def create_features(df):
    df = df.copy()

    mag_cols = ['u', 'g', 'r', 'i', 'z']

    # Basic color features
    df['u_g'] = df['u'] - df['g']
    df['g_r'] = df['g'] - df['r']
    df['r_i'] = df['r'] - df['i']
    df['i_z'] = df['i'] - df['z']
    df['u_r'] = df['u'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']

    # Statistical features
    df['flux_mean'] = df[mag_cols].mean(axis=1)
    df['flux_std'] = df[mag_cols].std(axis=1)
    df['flux_min'] = df[mag_cols].min(axis=1)
    df['flux_max'] = df[mag_cols].max(axis=1)
    df['flux_range'] = df['flux_max'] - df['flux_min']

    # Non-linear interactions (IMPORTANT for +2% boost)
    df['u_minus_z'] = df['u'] - df['z']
    df['r_over_g'] = df['r'] / (df['g'] + 1e-6)
    df['i_over_z'] = df['i'] / (df['z'] + 1e-6)

    df['color_index'] = (df['u'] - df['g']) - (df['r'] - df['i'])

    # Redshift features (if exists)
    if 'redshift' in df.columns:
        df['redshift_sq'] = df['redshift'] ** 2
        df['redshift_abs'] = np.abs(df['redshift'])
        df['redshift_log'] = np.log1p(np.abs(df['redshift']))

    # Positional interaction (if exists)
    if 'alpha' in df.columns and 'delta' in df.columns:
        df['alpha_delta'] = df['alpha'] * df['delta']

    return df


X = create_features(X)
test = create_features(test)

In [60]:
def make_numeric(df):
    df = df.copy()

    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str)

    return df

# 5. SAFE Encoding

In [62]:
from sklearn.preprocessing import OrdinalEncoder

cat_cols = X.select_dtypes(include=['object']).columns

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X[cat_cols] = encoder.fit_transform(X[cat_cols])
test[cat_cols] = encoder.transform(test[cat_cols])

# 6. Models

## XGBoost

In [54]:
xgb = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method='hist',
    eval_metric='mlogloss'
)




##  LightGBM

In [55]:
lgb = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.01,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8
)

## CatBoost

In [56]:

cat = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=6,
    loss_function='MultiClass',
    verbose=0
)

# 7. Stratified K-Fold Training

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_preds = np.zeros((len(test), len(np.unique(y))))
lgb_preds = np.zeros_like(xgb_preds)
cat_preds = np.zeros_like(xgb_preds)

# 8. Training Loop

In [63]:
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\nFOLD {fold+1}")

    X_train, X_valid = X.iloc[trn_idx], X.iloc[val_idx]
    y_train, y_valid = y[trn_idx], y[val_idx]

    # ---------------- XGB ----------------
    xgb.fit(X_train, y_train)
    pred = xgb.predict(X_valid)
    print("XGB:", balanced_accuracy_score(y_valid, pred))

    xgb_preds += xgb.predict_proba(test) / 5

    # ---------------- LGB ----------------
    lgb.fit(X_train, y_train)
    pred = lgb.predict(X_valid)
    print("LGB:", balanced_accuracy_score(y_valid, pred))

    lgb_preds += lgb.predict_proba(test) / 5

    # ---------------- CAT ----------------
    cat.fit(X_train, y_train)
    pred = cat.predict(X_valid)
    print("CAT:", balanced_accuracy_score(y_valid, pred))

    cat_preds += cat.predict_proba(test) / 5


FOLD 1
XGB: 0.9523993494304744


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:spectral_type: object, galaxy_population: object

# 9. ENSEMBLE

In [ ]:
final_probs = (
    xgb_preds +
    lgb_preds +
    cat_preds
) / 3

final_pred = np.argmax(final_probs, axis=1)
final_labels = le_y.inverse_transform(final_pred)

# 10. Submission File

In [39]:
submission = pd.DataFrame({
    "id": test["id"],
    "class": final_labels
})

submission.to_csv("submission.csv", index=False)


# 11. CREATE SUBMISSION




In [ ]:
submission=pd.DataFrame({

    'class':
    final_predictions

})

submission.to_csv(

    'submission.csv',

    index=False
)

print()
print(
    "submission.csv saved"
)